# Run Data100 PFG_MOEA_D Change G

Notebook nay chay rieng `PFG_MOEA_D` cho data100 voi `G = 10, 12, 14, 16, 18` trong ham `build_pfg(..., grid_size=G)`. Thoi gian chay cua moi instance duoc lay tu file `meta_info.csv` trong root cu `result_instance_5algo_with_stop_time_56`.

Voi `G=10`, notebook copy ket qua cu da co san tu root cu thay vi chay lai. Voi `G=12, 14, 16, 18`, notebook giu logic solver goc va chi thay gia tri `grid_size` khi tao PFG.


In [ ]:
import os
import sys
import time
import random
import shutil
from pathlib import Path
from typing import List, Set

import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_DIR = Path(r"D:\codePython\pythonProject\final_project\pfg_moead_vrpd_ver2")
os.chdir(PROJECT_DIR)
for path in [PROJECT_DIR.parent, PROJECT_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pfg_moead_vrpd_ver2.algorithm.pfg_moead_solver_stop import PFGMOEADSolverStop
from pfg_moead_vrpd_ver2.model.customer import Customer
from pfg_moead_vrpd_ver2.model.solution import Solution
from pfg_moead_vrpd_ver2.model.evaluator import Evaluator
from pfg_moead_vrpd_ver2.utils.pfg import build_pfg, sample_from_pfg

# CONFIG
GRID_SIZES = [10, 12, 14, 16, 18]
POP_SIZE = 100
NUM_TRUCKS = 6
SEED = 1
ALGO = "PFG_MOEA_D"

DATA_DIR = Path("data")
SOURCE_RESULT_ROOT = Path("result_instance_5algo_with_stop_time_56")
RESULT_ROOT = Path("result_instance_data100_PFG_MOEA_D_change_G")
RESULT_ROOT.mkdir(exist_ok=True)

# False = neu ket qua cua instance/G da ton tai thi bo qua, khong chay lai.
# True = bat buoc chay lai va ghi de ket qua cu.
FORCE_RERUN = False


In [ ]:
# LOAD DATASET + META HELPERS

def load_customers_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    customers = {}

    depot = df.iloc[0]
    customers[0] = Customer(
        cid=0,
        x=float(depot["x"]),
        y=float(depot["y"]),
        demand=0.0,
        ready_time=float(depot["open"]),
        due_time=float(depot["close"]),
        service_time=float(depot["servicetime"]),
        drone_serve=False,
        time=0.0,
    )

    for idx in range(1, len(df)):
        r = df.iloc[idx]
        customers[idx] = Customer(
            cid=idx,
            x=float(r["x"]),
            y=float(r["y"]),
            demand=float(r["demand"]),
            ready_time=float(r["open"]),
            due_time=float(r["close"]),
            service_time=float(r["servicetime"]),
            drone_serve=bool(r["drone_serve"]),
            time=float(r["time"]),
        )

    return customers


def read_source_max_time(instance_name):
    meta_file = SOURCE_RESULT_ROOT / f"result_{instance_name}" / "meta_info.csv"
    if not meta_file.exists():
        raise FileNotFoundError(f"Missing source meta_info.csv: {meta_file}")

    meta = pd.read_csv(meta_file)
    row = meta[meta["algorithm"] == ALGO]
    if row.empty:
        raise ValueError(f"Algorithm {ALGO} not found in {meta_file}")

    return float(row.iloc[0]["max_time"]), meta_file


def output_dir_for(grid_size, instance_name):
    return RESULT_ROOT / f"G_{grid_size}" / f"result_{instance_name}"


In [ ]:
# SOLVER CHANGE G
# Goc PFG_MOEA_D dung build_pfg(self.external_pop), tuc grid_size mac dinh = 10.
# Lop nay giu nguyen run loop cua pfg_moead_solver_stop.py, chi truyen grid_size=self.grid_size.

class PFGMOEADSolverStopChangeG(PFGMOEADSolverStop):
    def __init__(self, *args, grid_size=10, **kwargs):
        super().__init__(*args, **kwargs)
        self.grid_size = grid_size

    def run(self):
        start_time = time.time()

        self.external_pop = []
        self.population = []
        self.population_perms = []
        self.population_drones = []

        self._init_population()
        self._update_ideal_point()

        for s in self.population:
            self._update_external_population(s)

        generation = 0
        no_improve = 0

        while (self.max_time is None or time.time() - start_time < self.max_time) and no_improve < self.no_improve_limit:
            ep_before = set((s.makespan, s.carbonEmission) for s in self.external_pop)

            ep = build_pfg(self.external_pop, grid_size=self.grid_size)
            self.pfg_pool = ep

            for i in range(self.pop_size):
                if self.max_time is not None and time.time() - start_time >= self.max_time:
                    break

                p1 = self._select_from_neighborhood(i)

                if self.rnd.random() > self.crossover_prob:
                    continue

                perm1 = self.population_perms[p1]
                drone1 = self.population_drones[p1]

                pfg_parent = sample_from_pfg(self.pfg_pool, self.rnd)
                perm2 = self._flatten(pfg_parent.truckRoutes)
                drone2 = pfg_parent.droneCustomers

                child_perm = self._order_crossover_safe(perm1, perm2)

                if self.rnd.random() < self.mutation_swap_prob:
                    self._swap_mutate(child_perm)

                child = Solution()
                child.truckRoutes = self._split(child_perm, self.num_trucks)

                child_drones: List[Set[int]] = []
                for t in range(self.num_trucks):
                    d = self._uniform_crossover_drone(drone1[t], drone2[t], child.truckRoutes[t])
                    self._flip_mutate_drone(d, child.truckRoutes[t])
                    child_drones.append(d)

                child.droneCustomers = child_drones
                child.normalize()
                self._enforce_no_adjacent_drones(child)

                self._local_search(child)

                Evaluator.evaluate(child, self.customers)

                self._update_ideal(child)
                self._update_neighborhood(i, child)
                self._update_external_population(child)

            generation += 1

            ep_after = set((s.makespan, s.carbonEmission) for s in self.external_pop)

            if ep_after != ep_before:
                no_improve = 0
            else:
                no_improve += 1

        self.generations = generation
        self.run_time = time.time() - start_time
        return self.external_pop[:]


In [ ]:
# SAVE RESULT HELPERS

def print_and_save_pareto(pf, algo_name, result_dir):
    print("\n" + "=" * 80)
    print(f"{algo_name} Pareto Front")
    print(f"Pareto size = {len(pf)}")
    print("=" * 80)

    rows = []

    for i, s in enumerate(pf, 1):
        print(f"\n--- Solution {i} ---")
        print(f"Makespan = {s.makespan:.4f}")
        print(f"Carbon   = {s.carbonEmission:.4f}")

        for t, route in enumerate(s.truckRoutes):
            drone_list = []
            if t < len(s.droneCustomers):
                drone_list = sorted(list(s.droneCustomers[t]))

            print(f"Truck {t + 1}: {route}")
            print(f"Drone {t + 1}: {drone_list}")

            rows.append({
                "algorithm": algo_name,
                "solution_id": i,
                "truck_id": t + 1,
                "makespan": s.makespan,
                "carbon": s.carbonEmission,
                "truck_route": " ".join(map(str, route)),
                "drone_customers": " ".join(map(str, drone_list)),
            })

    df = pd.DataFrame(rows)
    csv_path = result_dir / f"pareto_{algo_name}.csv"
    df.to_csv(csv_path, index=False)
    print("\nSaved Pareto tours to:", csv_path)


def save_meta_info(result_dir, grid_size, max_time, run_time, generations, source_meta_file):
    df_meta = pd.DataFrame([
        {
            "algorithm": ALGO,
            "pop_size": POP_SIZE,
            "G": grid_size,
            "grid_size": grid_size,
            "max_time": max_time,
            "run_time": run_time,
            "generations": generations,
            "source_meta_file": str(source_meta_file),
        }
    ])

    meta_path = result_dir / "meta_info.csv"
    df_meta.to_csv(meta_path, index=False)
    print("Saved meta info to:", meta_path)


def copy_g10_from_source(instance_name):
    source_dir = SOURCE_RESULT_ROOT / f"result_{instance_name}"
    result_dir = output_dir_for(10, instance_name)
    result_dir.mkdir(parents=True, exist_ok=True)

    source_pareto = source_dir / f"pareto_{ALGO}.csv"
    source_meta = source_dir / "meta_info.csv"
    target_pareto = result_dir / f"pareto_{ALGO}.csv"
    target_meta = result_dir / "meta_info.csv"

    if target_pareto.exists() and target_meta.exists() and not FORCE_RERUN:
        print("G=10 result already exists, skip copy:", result_dir)
        return

    if not source_pareto.exists():
        raise FileNotFoundError(f"Missing source Pareto CSV: {source_pareto}")
    if not source_meta.exists():
        raise FileNotFoundError(f"Missing source meta_info.csv: {source_meta}")

    shutil.copy2(source_pareto, target_pareto)

    meta = pd.read_csv(source_meta)
    meta = meta[meta["algorithm"] == ALGO].copy()
    if meta.empty:
        raise ValueError(f"Algorithm {ALGO} not found in {source_meta}")

    meta["pop_size"] = POP_SIZE
    meta["G"] = 10
    meta["grid_size"] = 10
    meta["source_meta_file"] = str(source_meta)
    meta.to_csv(target_meta, index=False)

    print("Copied G=10 Pareto to:", target_pareto)
    print("Copied G=10 meta to:", target_meta)


In [ ]:
# RUN ONE INSTANCE FOR ONE G

def run_instance_with_g(instance_name, customers, grid_size):
    result_dir = output_dir_for(grid_size, instance_name)
    pareto_file = result_dir / f"pareto_{ALGO}.csv"
    meta_file = result_dir / "meta_info.csv"

    print(f"\n========== INSTANCE {instance_name} | G={grid_size} ==========")

    if grid_size == 10:
        copy_g10_from_source(instance_name)
        return

    if pareto_file.exists() and meta_file.exists() and not FORCE_RERUN:
        print("Result already exists, skip:", result_dir)
        return

    result_dir.mkdir(parents=True, exist_ok=True)

    max_time, source_meta_file = read_source_max_time(instance_name)
    print("Source meta:", source_meta_file)
    print("Use max_time:", max_time)

    random.seed(SEED)
    np.random.seed(SEED)

    solver = PFGMOEADSolverStopChangeG(
        pop_size=POP_SIZE,
        max_time=max_time,
        num_trucks=NUM_TRUCKS,
        customers=customers,
        seed=SEED,
        grid_size=grid_size,
    )

    start = time.time()
    pf = solver.run()
    run_time = time.time() - start

    print_and_save_pareto(pf, ALGO, result_dir)
    save_meta_info(
        result_dir=result_dir,
        grid_size=grid_size,
        max_time=max_time,
        run_time=run_time,
        generations=getattr(solver, "generations", None),
        source_meta_file=source_meta_file,
    )


In [ ]:
# MAIN LOOP
INSTANCE_FILES = sorted([
    f for f in DATA_DIR.iterdir()
    if f.is_file() and f.suffix.lower() == ".csv"
])

print("Total instances:", len(INSTANCE_FILES))
print("G values:", GRID_SIZES)

for file_path in tqdm(INSTANCE_FILES):
    instance_name = file_path.stem
    customers = load_customers_from_csv(file_path)

    for grid_size in GRID_SIZES:
        run_instance_with_g(instance_name, customers, grid_size)
